In [4]:
import pandas as pd
import os


def process_sdcis_to_parquet_streaming(
    input_folder, output_folder, chunk_size=500000, internal_chunk_size=10000
):
    """
    Process SDCI CSV files from a folder and save them into Parquet files in chunks.

    Parameters:
    - input_folder: Path to the folder containing SDCI CSV files.
    - output_folder: Path to save the Parquet files.
    - chunk_size: Number of rows per output Parquet file.
    - internal_chunk_size: Number of rows to read at a time from each CSV.
    """
    os.makedirs(output_folder, exist_ok=True)

    current_chunk = []
    current_count = 0
    chunk_number = 0

    # Get all relevant CSV files
    csv_files = [
        f for f in os.listdir(input_folder) if f.endswith("_solos_patterns.csv")
    ]

    for file in csv_files:
        file_path = os.path.join(input_folder, file)
        # Read CSV in chunks
        reader = pd.read_csv(file_path, chunksize=internal_chunk_size)

        for df_chunk in reader:
            # Rename 'Sector' to 'sector' to match SDCI class attributes
            df_chunk.rename(columns={"Sector": "sector"}, inplace=True)

            # Add this chunk to the current batch
            current_chunk.append(df_chunk)
            current_count += len(df_chunk)

            # If we've reached the chunk size, save to Parquet
            if current_count >= chunk_size:
                combined_chunk = pd.concat(current_chunk, ignore_index=True)
                output_file = os.path.join(
                    output_folder, f"sdcis_chunk_{chunk_number}.parquet"
                )
                combined_chunk.to_parquet(output_file)
                print(f"Saved chunk {chunk_number} with {len(combined_chunk)} rows")

                # Reset for next chunk
                current_chunk = []
                current_count = 0
                chunk_number += 1

    # Save any remaining data
    if current_count > 0:
        combined_chunk = pd.concat(current_chunk, ignore_index=True)
        output_file = os.path.join(output_folder, f"sdcis_chunk_{chunk_number}.parquet")
        combined_chunk.to_parquet(output_file)
        print(f"Saved chunk {chunk_number} with {len(combined_chunk)} rows")


# Example usage
input_folder = r"/workspace/qgi\qgi-4.0\data\countries_sdci"  # r"/workspaces/qgi-4.0/data/countries_sdci"
output_folder = r"/workspace/qgi\qgi-4.0\data\countries_sdci_parquet"  # r"/workspaces/qgi-4.0/data/countries_sdci_parquet"
process_sdcis_to_parquet_streaming(input_folder, output_folder)

/workspace\AppData\Local\Temp\ipykernel_7484\2521469355.py:38: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_chunk = pd.concat(current_chunk, ignore_index=True)


Saved chunk 0 with 506587 rows


/workspace\AppData\Local\Temp\ipykernel_7484\2521469355.py:38: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_chunk = pd.concat(current_chunk, ignore_index=True)


Saved chunk 1 with 508052 rows


/workspace\AppData\Local\Temp\ipykernel_7484\2521469355.py:38: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_chunk = pd.concat(current_chunk, ignore_index=True)


Saved chunk 2 with 506693 rows


/workspace\AppData\Local\Temp\ipykernel_7484\2521469355.py:50: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_chunk = pd.concat(current_chunk, ignore_index=True)


Saved chunk 3 with 443169 rows


In [ ]:
import pandas as pd
import os

## TODO revisit this and fix it


def process_patterns_to_parquet_streaming(
    input_folder, output_folder, chunk_size=500000, internal_chunk_size=10000
):
    """
    Process country pattern CSV files from a folder and save them into Parquet files in chunks.
    """
    os.makedirs(output_folder, exist_ok=True)

    current_chunk = []
    current_count = 0
    chunk_number = 0

    # Get all relevant CSV files
    csv_files = [f for f in os.listdir(input_folder) if f.endswith("_patterns.csv")]

    for file in csv_files:
        file_path = os.path.join(input_folder, file)
        # Read CSV in chunks
        reader = pd.read_csv(file_path, chunksize=internal_chunk_size)

        for df_chunk in reader:
            # Define required columns and their types
            type_mapping = {
                "country_a_id_fk": "int64",
                "country_b_id_fk": "int64",
                "start_year_a_fk": "int64",
                "start_year_b_fk": "int64",
                "pattern_length_fk": "int64",
                "indexes": "int64",
                "avg_corr": "float64",
                "orgs": "int64",
                "Power": "float64",
                "similar_event_year_a": "object",
                "similar_raw_event": "object",
                "similar_event_year_b": "object",
            }

            # Convert columns that exist in the current chunk
            for column, dtype in type_mapping.items():
                if column in df_chunk.columns:
                    # Handle 'object' type columns separately
                    if dtype == "object":
                        df_chunk[column] = df_chunk[column].astype(str)
                    else:
                        # Use pandas' safe conversion method
                        df_chunk[column] = pd.to_numeric(
                            df_chunk[column], errors="coerce"
                        ).astype(dtype)

            # Add this chunk to the current batch
            current_chunk.append(df_chunk)
            current_count += len(df_chunk)

            # If we've reached the chunk size, save to Parquet
            if current_count >= chunk_size:
                combined_chunk = pd.concat(current_chunk, ignore_index=True)
                output_file = os.path.join(
                    output_folder, f"patterns_chunk_{chunk_number}.parquet"
                )
                combined_chunk.to_parquet(output_file)
                print(f"Saved chunk {chunk_number} with {len(combined_chunk)} rows")

                # Reset for next chunk
                current_chunk = []
                current_count = 0
                chunk_number += 1

    # Save any remaining data
    if current_count > 0:
        combined_chunk = pd.concat(current_chunk, ignore_index=True)
        output_file = os.path.join(
            output_folder, f"patterns_chunk_{chunk_number}.parquet"
        )
        combined_chunk.to_parquet(output_file)
        print(f"Saved chunk {chunk_number} with {len(combined_chunk)} rows")


# Example usage
input_folder = r"/workspace/qgi\qgi-4.0\data\country_patterns"
output_folder = r"/workspace/qgi\qgi-4.0\data\country_patterns_parquet"
process_patterns_to_parquet_streaming(input_folder, output_folder)

/workspace\AppData\Local\Temp\ipykernel_31372\1613484041.py:67: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_chunk = pd.concat(current_chunk, ignore_index=True)


Saved chunk 0 with 461814 rows
